# Train a plant-disease YOLOv8 (AR glasses project, Step 7.6)

Trains YOLOv8 on **PlantDoc** (29 classes: corn, tomato, potato, apple, grape... leaves and diseases)
so the AR backend can run it on its fast YOLOv8 C++ path (~150-400 ms instead of ~2 s).

**Before running:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.
Takes about 45-60 min. Keep this tab open. At the end, `plant_yolov8_results.zip` is downloaded:
put `best.pt` from it into the project's `models/` folder.

Training uses PlantDoc **TRAIN** only; PlantDoc **TEST** is the validation set.

In [ ]:
# 1. Check that a GPU is attached (should show a Tesla T4)
!nvidia-smi

In [ ]:
# 2. Install Ultralytics (YOLOv8)
!pip -q install ultralytics
import ultralytics; ultralytics.checks()

In [ ]:
# 3. Download PlantDoc (~1 GB)
!git clone -q --depth 1 https://github.com/pratikkayal/PlantDoc-Object-Detection-Dataset plantdoc
!ls plantdoc

In [ ]:
%%writefile plantdoc_to_yolo.py
"""
Step 7.6: convert the PlantDoc dataset into a YOLO training set.

Same script on this PC and on Google Colab:
  python plantdoc_to_yolo.py --src <PlantDoc clone> --out <output folder> [--max-side 1280]

PlantDoc (https://github.com/pratikkayal/PlantDoc-Object-Detection-Dataset):
  TRAIN/, TEST/                images (some file names contain ? : & ... - not valid on Windows)
  train_labels.csv, test_labels.csv   filename,width,height,class,xmin,ymin,xmax,ymax

Output (Ultralytics YOLO format):
  images/train/train_00001.jpg ...   SAFE new names, JPEG, longest side <= --max-side
  labels/train/train_00001.txt ...   "class cx cy w h", normalized 0..1
  images/val, labels/val             from PlantDoc TEST (the standard benchmark split)
  data.yaml                          classes in a fixed (sorted) order
  mapping.csv                        new name -> original PlantDoc name (traceability)

TRAIN only is used for training; TEST is the validation set, and also where our 45 evaluation
leaves (testdata/plant/plantdoc) come from - so the comparison with the old model stays fair.
"""
import argparse
import collections
import csv
import os
import sys

from PIL import Image


def read_labels(path):
    rows = collections.defaultdict(list)
    with open(path, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            rows[r["filename"]].append(r)
    return rows


def convert(src, out, split_in, split_out, csv_name, classes, max_side, stats, mapping):
    labels = read_labels(os.path.join(src, csv_name))
    os.makedirs(os.path.join(out, "images", split_out), exist_ok=True)
    os.makedirs(os.path.join(out, "labels", split_out), exist_ok=True)
    idx = 0
    for fname in sorted(labels):
        p = os.path.join(src, split_in, fname)
        if not os.path.exists(p):
            stats[f"{split_out}: image file missing"] += 1
            continue
        try:
            im = Image.open(p)
            im.load()
            im = im.convert("RGB")
        except Exception:
            stats[f"{split_out}: image unreadable"] += 1
            continue
        lines = []
        for r in labels[fname]:
            # boxes refer to the size written in the CSV (the annotation frame)
            w, h = float(r["width"]), float(r["height"])
            if w <= 0 or h <= 0:
                w, h = float(im.width), float(im.height)
            x1, y1 = max(0.0, float(r["xmin"]) / w), max(0.0, float(r["ymin"]) / h)
            x2, y2 = min(1.0, float(r["xmax"]) / w), min(1.0, float(r["ymax"]) / h)
            if x2 <= x1 or y2 <= y1:
                stats[f"{split_out}: empty box skipped"] += 1
                continue
            lines.append(f"{classes.index(r['class'])} {(x1 + x2) / 2:.6f} {(y1 + y2) / 2:.6f} "
                         f"{x2 - x1:.6f} {y2 - y1:.6f}")
        if not lines:
            stats[f"{split_out}: no usable box"] += 1
            continue
        idx += 1
        name = f"{split_out}_{idx:05d}"
        s = max_side / max(im.width, im.height)
        if s < 1:
            im = im.resize((round(im.width * s), round(im.height * s)), Image.LANCZOS)
        im.save(os.path.join(out, "images", split_out, name + ".jpg"), quality=92)
        with open(os.path.join(out, "labels", split_out, name + ".txt"), "w") as f:
            f.write("\n".join(lines) + "\n")
        mapping.append((f"{split_out}/{name}.jpg", f"{split_in}/{fname}"))
        stats[f"{split_out}: images"] += 1
        stats[f"{split_out}: boxes"] += len(lines)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", required=True, help="PlantDoc clone (with TRAIN/, TEST/, *_labels.csv)")
    ap.add_argument("--out", required=True, help="output folder for the YOLO dataset")
    ap.add_argument("--max-side", type=int, default=1280, help="shrink images so the longest side is <= this")
    args = ap.parse_args()

    classes = sorted({r["class"] for name in ("train_labels.csv", "test_labels.csv")
                      for rows in read_labels(os.path.join(args.src, name)).values() for r in rows})
    stats, mapping = collections.Counter(), []
    convert(args.src, args.out, "TRAIN", "train", "train_labels.csv", classes, args.max_side, stats, mapping)
    convert(args.src, args.out, "TEST", "val", "test_labels.csv", classes, args.max_side, stats, mapping)

    out_abs = os.path.abspath(args.out)
    with open(os.path.join(args.out, "data.yaml"), "w", encoding="utf-8") as f:
        f.write(f"path: {out_abs}\ntrain: images/train\nval: images/val\n")
        f.write(f"nc: {len(classes)}\nnames:\n")
        for i, c in enumerate(classes):
            f.write(f"  {i}: \"{c}\"\n")
    with open(os.path.join(args.out, "mapping.csv"), "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerows([("yolo_image", "plantdoc_image"), *mapping])

    print(f"{len(classes)} classes: {classes}")
    for k in sorted(stats):
        print(f"  {k}: {stats[k]}")
    print(f"wrote {out_abs}/data.yaml")
    if stats["train: images"] < 2000 or stats["val: images"] < 200:
        sys.exit("too few images converted - check the source folder")


if __name__ == "__main__":
    main()

In [ ]:
# 4. Convert to YOLO format (safe file names, images shrunk to <= 1280 px)
!python plantdoc_to_yolo.py --src plantdoc --out plantdoc-yolo

## Training settings
`MODEL`: `yolov8s.pt` (more accurate, ~2-3x slower on the laptop) or `yolov8n.pt` (fastest).
`yolov8s` is the default: a "look and hold" check can afford ~300-400 ms, accuracy matters more.

In [ ]:
# 5. Train
from ultralytics import YOLO
MODEL = "yolov8s.pt"
EPOCHS = 100
model = YOLO(MODEL)
results = model.train(data="plantdoc-yolo/data.yaml", epochs=EPOCHS, imgsz=640, batch=16, patience=30,
                      project="/content/runs", name="plant-yolov8", exist_ok=True, plots=True, seed=0)
print("best weights:", model.trainer.best)

In [ ]:
# 6. Validate the best weights on PlantDoc TEST (per-class numbers)
best = YOLO(model.trainer.best)
m = best.val(data="plantdoc-yolo/data.yaml", imgsz=640, split="val", plots=True)
print(f"mAP50 = {m.box.map50:.3f}   mAP50-95 = {m.box.map:.3f}")
for i, name in best.names.items():
    if i < len(m.box.ap50):
        print(f"  {name:38} AP50 {m.box.ap50[i]:.3f}")

In [ ]:
# 7. Download the results (best.pt + training curves + confusion matrix)
import os, shutil, glob
from google.colab import files
out = "/content/plant_yolov8_results"
os.makedirs(out, exist_ok=True)
shutil.copy(model.trainer.best, os.path.join(out, "best.pt"))
for f in glob.glob("/content/runs/plant-yolov8/*.png") + glob.glob("/content/runs/plant-yolov8/*.csv"):
    shutil.copy(f, out)
shutil.make_archive("/content/plant_yolov8_results", "zip", out)
files.download("/content/plant_yolov8_results.zip")